# MicroCLIP — Colab A100 Training

Runs any config from the ablation matrix on a Colab A100.

**Setup:** Runtime → Change runtime type → **A100 GPU**.

**Preemption-proof:** `runs/` and `artifacts/` are symlinked to Google Drive, so
checkpoints and the tokenizer survive session death. If the session is preempted,
just re-run the whole notebook — training auto-resumes from `last.pt`
(COCO re-downloads to the local SSD, ~10 min).

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > A100"
name = torch.cuda.get_device_name(0)
print(name, "| bf16:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    print("WARNING: not an A100 — configs assume Ampere+ (bf16 AMP, batch 512)")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

PERSIST = "/content/drive/MyDrive/microclip"
os.makedirs(f"{PERSIST}/runs", exist_ok=True)
os.makedirs(f"{PERSIST}/artifacts/tokenizer", exist_ok=True)

In [ ]:
%cd /content
!git clone https://github.com/umutonuryasar/microclip.git 2>/dev/null || git -C microclip pull
%cd /content/microclip
!pip -q install -e .

In [ ]:
import os

# Checkpoints + tokenizer live on Drive (survive preemption).
# COCO images stay on the local SSD — Drive I/O is far too slow for training.
for link, target in [("runs", f"{PERSIST}/runs"), ("artifacts", f"{PERSIST}/artifacts")]:
    if not os.path.islink(link):
        os.system(f"rm -rf {link}")
        os.symlink(target, link)
print("runs/ ->", os.readlink("runs"))
print("artifacts/ ->", os.readlink("artifacts"))

In [ ]:
%%bash
# COCO 2017 to local SSD (~20 GB total). Skips anything already unzipped.
mkdir -p data/coco && cd data/coco
for z in train2017 val2017; do
  if [ ! -d $z ]; then
    wget -q -c http://images.cocodataset.org/zips/$z.zip
    unzip -q $z.zip && rm $z.zip
  fi
done
if [ ! -d annotations ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
fi
du -sh *

In [ ]:
import os

# One-off; lands on Drive via the artifacts/ symlink.
if not os.path.exists("artifacts/tokenizer/bpe16k.json"):
    !python scripts/train_tokenizer.py --config configs/base.yml
else:
    print("tokenizer exists — skipping")

In [ ]:
# Sanity check on the real data path (~few min on A100) before burning GPU hours.
# Safe to comment out on later sessions of the same clone.
!python scripts/smoke_test.py --config configs/base.yml


In [ ]:
import wandb

wandb.login()

## Train

Pick a config from the ablation matrix:

| Run | Config |
|---|---|
| Main: sigmoid / softmax @ b512 | `configs/sigmoid_b512.yml` / `configs/softmax_b512.yml` |
| Batch-size ablations | `configs/ablations/{sigmoid,softmax}_b{128,256}.yml` |
| Optimizer / LR / init / ViT | `configs/ablations/{optimizer_sgd,lr_constant,init_xavier,init_he,vit_tiny}.yml` |

Re-running this cell after preemption resumes from `runs/<run_name>/last.pt` automatically.

In [ ]:
CONFIG = "configs/sigmoid_b512.yml"

# num_workers=8: config default (4) targets the local dev box; Colab A100 VMs have ~12 vCPUs.
!python scripts/train.py --config $CONFIG --set data.num_workers=8